# Day 2 FOXF1 / BMP4 reporters — 02_reporter_pixel_quantification

**Feeds:** quality control only; notebook `03` recomputes the same cutoffs itself

**Position in the chain:** run `00`, `01`, `02`, `03` in order, from this lane's directory (`imaging/foxf1_bmp4_day2_fig2g/`).

Ported from the original analysis `FOXF1_BMP4_day2_expression/notebooks/02_reporter_pixel_quantification.ipynb`.

**Changes from the original notebook**
1. No code cell of the original was edited.
2. The notebook ships without outputs, as the original notebook file does.
3. CZI files are read through `src/trunk_morph_ref/czi_compat.py` instead of `czifile`, a change of one import in `scripts/day2_quantification_helpers.py`.


            # 02 | Reporter Signal And Threshold Review

            ## Notebook Scope

            This notebook now shows only the current threshold logic used by the active pipeline:

            - pool off-cyst raw pixels across all z planes within each image
            - subtract the per-image off-cyst median
            - pool corrected off-cyst pixels across all images
            - learn one global `q99.5` cutoff per channel
            - overlay those cutoffs back onto the selected review plane for each stack

            There are no competing threshold families in this notebook anymore.
            

In [ ]:
import os
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
if (CWD / "scripts").exists() and (CWD / "results").exists():
    ROOT = CWD
elif (CWD.parent / "scripts").exists() and (CWD.parent / "results").exists():
    ROOT = CWD.parent.resolve()
else:
    ROOT = CWD

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)
print("Python:", sys.executable)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from IPython.display import display

from scripts import day2_quantification_helpers as dqh

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 200)

            ## Paths And Parameters

            This notebook reads the mask outputs already written by notebook `01`.
            It does not run its own alternate thresholding pipeline.
            

In [ ]:
MANIFEST_OUTPUT = ROOT / "results" / "manifests" / "raw_input_manifest.tsv"
PLANE_METRICS_OUTPUT = ROOT / "results" / "tables" / "01_mask_and_plane_metrics.tsv"

QC_DIR = ROOT / "results" / "qc" / "02_threshold_review"
QC_DIR.mkdir(parents=True, exist_ok=True)
CURRENT_Q995_OVERLAY_FIG = QC_DIR / "median_subtracted_global_q995_review_plane_overlays.png"
CURRENT_Q995_BG_HIST_FIG = QC_DIR / "median_subtracted_global_q995_background_histograms.png"
CURRENT_Q995_MASKED_HIST_FIG = QC_DIR / "median_subtracted_global_q995_within_cyst_histograms.png"

BACKGROUND_ESTIMATOR = "whole_off_organoid"
BACKGROUND_ANNULUS_INNER_PX = 6
BACKGROUND_ANNULUS_OUTER_PX = 20
BACKGROUND_MIN_RING_PIXELS = 1000
CURRENT_Q995 = 0.995

In [ ]:
manifest_df = pd.read_csv(MANIFEST_OUTPUT, sep="\t")
plane_metrics_df = pd.read_csv(PLANE_METRICS_OUTPUT, sep="\t")
manifest_df = manifest_df.sort_values("file_id").reset_index(drop=True)
plane_metrics_df = plane_metrics_df.sort_values(["file_id", "z_index"]).reset_index(drop=True)

display(manifest_df[["file_id", "position_label", "file_name", "selected_review_z_1based", "size_z"]])

            ## Current Global `q99.5` Thresholds

            These thresholds are the only active ones now.
            

In [ ]:
def compute_current_q995_thresholds():
    per_file_bg_medians: dict[int, dict[str, float]] = {}
    fox_bg_corrected_chunks: list[np.ndarray] = []
    bmp_bg_corrected_chunks: list[np.ndarray] = []
    fox_masked_corrected_chunks: list[np.ndarray] = []
    bmp_masked_corrected_chunks: list[np.ndarray] = []
    summary_rows: list[dict[str, object]] = []

    for row in manifest_df.itertuples(index=False):
        stack = dqh.load_czi_stack(ROOT / row.file_path)
        metric_sub = plane_metrics_df.loc[plane_metrics_df["file_id"] == int(row.file_id)].sort_values("z_index")
        fox_idx = dqh.channel_index(stack.canonical_channel_names, "foxf1")
        bmp_idx = dqh.channel_index(stack.canonical_channel_names, "bmp4")

        fox_bg_raw_chunks: list[np.ndarray] = []
        bmp_bg_raw_chunks: list[np.ndarray] = []

        for metric in metric_sub.itertuples(index=False):
            mask = tifffile.imread(ROOT / metric.mask_path).astype(bool)
            z_index = int(metric.z_index)
            bg_mask = dqh.background_reference_mask(
                organoid_mask=mask,
                background_estimator=BACKGROUND_ESTIMATOR,
                annulus_inner_radius_px=BACKGROUND_ANNULUS_INNER_PX,
                annulus_outer_radius_px=BACKGROUND_ANNULUS_OUTER_PX,
                min_ring_pixels=BACKGROUND_MIN_RING_PIXELS,
            )
            fox_raw = np.asarray(stack.data_czyx[fox_idx, z_index], dtype=np.float32)
            bmp_raw = np.asarray(stack.data_czyx[bmp_idx, z_index], dtype=np.float32)
            fox_bg_valid = np.asarray(bg_mask, dtype=bool) & np.isfinite(fox_raw)
            bmp_bg_valid = np.asarray(bg_mask, dtype=bool) & np.isfinite(bmp_raw)
            if np.any(fox_bg_valid):
                fox_bg_raw_chunks.append(np.asarray(fox_raw[fox_bg_valid], dtype=np.float32))
            if np.any(bmp_bg_valid):
                bmp_bg_raw_chunks.append(np.asarray(bmp_raw[bmp_bg_valid], dtype=np.float32))

        fox_bg_raw = np.concatenate(fox_bg_raw_chunks).astype(np.float32)
        bmp_bg_raw = np.concatenate(bmp_bg_raw_chunks).astype(np.float32)
        fox_median = float(np.median(fox_bg_raw))
        bmp_median = float(np.median(bmp_bg_raw))
        per_file_bg_medians[int(row.file_id)] = {"foxf1": fox_median, "bmp4": bmp_median}

        for metric in metric_sub.itertuples(index=False):
            mask = tifffile.imread(ROOT / metric.mask_path).astype(bool)
            z_index = int(metric.z_index)
            fox_raw = np.asarray(stack.data_czyx[fox_idx, z_index], dtype=np.float32)
            bmp_raw = np.asarray(stack.data_czyx[bmp_idx, z_index], dtype=np.float32)
            fox_corr = fox_raw - np.float32(fox_median)
            bmp_corr = bmp_raw - np.float32(bmp_median)
            fox_masked_valid = np.asarray(mask, dtype=bool) & np.isfinite(fox_corr)
            bmp_masked_valid = np.asarray(mask, dtype=bool) & np.isfinite(bmp_corr)
            if np.any(fox_masked_valid):
                fox_masked_corrected_chunks.append(np.asarray(fox_corr[fox_masked_valid], dtype=np.float32))
            if np.any(bmp_masked_valid):
                bmp_masked_corrected_chunks.append(np.asarray(bmp_corr[bmp_masked_valid], dtype=np.float32))

        fox_bg_corrected = fox_bg_raw - np.float32(fox_median)
        bmp_bg_corrected = bmp_bg_raw - np.float32(bmp_median)
        fox_bg_corrected_chunks.append(fox_bg_corrected)
        bmp_bg_corrected_chunks.append(bmp_bg_corrected)
        summary_rows.append(
            {
                "file_id": int(row.file_id),
                "position_label": str(row.position_label),
                "size_z": int(row.size_z),
                "selected_review_z_1based": int(row.selected_review_z_1based),
                "foxf1_bg_median_raw": fox_median,
                "bmp4_bg_median_raw": bmp_median,
                "foxf1_off_cyst_pixels": int(fox_bg_raw.size),
                "bmp4_off_cyst_pixels": int(bmp_bg_raw.size),
            }
        )

    fox_bg_corrected_all = np.concatenate(fox_bg_corrected_chunks).astype(np.float32)
    bmp_bg_corrected_all = np.concatenate(bmp_bg_corrected_chunks).astype(np.float32)
    fox_masked_corrected_all = np.concatenate(fox_masked_corrected_chunks).astype(np.float32)
    bmp_masked_corrected_all = np.concatenate(bmp_masked_corrected_chunks).astype(np.float32)
    fox_threshold = float(np.quantile(fox_bg_corrected_all, CURRENT_Q995))
    bmp_threshold = float(np.quantile(bmp_bg_corrected_all, CURRENT_Q995))
    threshold_table = pd.DataFrame(
        {
            "channel_key": ["foxf1", "bmp4"],
            "global_q995_threshold_corrected": [fox_threshold, bmp_threshold],
        }
    )
    summary_df = pd.DataFrame(summary_rows).sort_values("file_id").reset_index(drop=True)
    return (
        per_file_bg_medians,
        fox_threshold,
        bmp_threshold,
        threshold_table,
        summary_df,
        fox_bg_corrected_all,
        bmp_bg_corrected_all,
        fox_masked_corrected_all,
        bmp_masked_corrected_all,
    )


(
    CURRENT_BG_MEDIANS,
    CURRENT_FOXF1_Q995,
    CURRENT_BMP4_Q995,
    CURRENT_Q995_THRESHOLD_DF,
    CURRENT_BG_SUMMARY_DF,
    CURRENT_FOX_BG_ALL,
    CURRENT_BMP_BG_ALL,
    CURRENT_FOX_MASKED_ALL,
    CURRENT_BMP_MASKED_ALL,
) = compute_current_q995_thresholds()
display(CURRENT_Q995_THRESHOLD_DF)
display(CURRENT_BG_SUMMARY_DF)

In [ ]:
def _background_xlim(values: np.ndarray, threshold: float) -> tuple[float, float]:
    arr = np.asarray(values, dtype=np.float32)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return (-50.0, 50.0)
    lo = float(np.quantile(arr, 0.0001))
    hi = float(np.quantile(arr, 0.9999))
    hi = max(hi, float(threshold) * 1.35)
    span = hi - lo
    pad = max(20.0, 0.12 * span)
    return (lo - pad, hi + pad)


fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.2), constrained_layout=True)

axes[0].hist(CURRENT_FOX_BG_ALL, bins=180, color="#ef9a9a", edgecolor="none", alpha=0.85, density=True, label="pooled off-cyst background")
axes[0].axvline(0.0, color="#607d8b", linewidth=1.0, linestyle="--", label="0 after median subtraction")
axes[0].axvline(CURRENT_FOXF1_Q995, color="#111111", linewidth=1.2, linestyle="--", label="global q99.5 cutoff")
axes[0].set_title("FOXF1 pooled off-cyst corrected background")
axes[0].set_xlabel("FOXF1 corrected intensity")
axes[0].set_ylabel("density")
axes[0].set_xlim(*_background_xlim(CURRENT_FOX_BG_ALL, CURRENT_FOXF1_Q995))
axes[0].legend(frameon=False, fontsize=8)

axes[1].hist(CURRENT_BMP_BG_ALL, bins=180, color="#a5d6a7", edgecolor="none", alpha=0.85, density=True, label="pooled off-cyst background")
axes[1].axvline(0.0, color="#607d8b", linewidth=1.0, linestyle="--", label="0 after median subtraction")
axes[1].axvline(CURRENT_BMP4_Q995, color="#111111", linewidth=1.2, linestyle="--", label="global q99.5 cutoff")
axes[1].set_title("BMP4 pooled off-cyst corrected background")
axes[1].set_xlabel("BMP4 corrected intensity")
axes[1].set_ylabel("density")
axes[1].set_xlim(*_background_xlim(CURRENT_BMP_BG_ALL, CURRENT_BMP4_Q995))
axes[1].legend(frameon=False, fontsize=8)

fig.savefig(CURRENT_Q995_BG_HIST_FIG, dpi=220, bbox_inches="tight")
print("Saved:", CURRENT_Q995_BG_HIST_FIG)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.2), constrained_layout=True)

axes[0].hist(CURRENT_FOX_MASKED_ALL, bins=180, color="#ef9a9a", edgecolor="none", alpha=0.85, density=True, label="pooled within-cyst pixels")
axes[0].axvline(CURRENT_FOXF1_Q995, color="#111111", linewidth=1.2, linestyle="--", label="global q99.5 cutoff")
axes[0].set_title("FOXF1 pooled within-cyst corrected pixels")
axes[0].set_xlabel("FOXF1 corrected intensity")
axes[0].set_ylabel("density")
axes[0].set_xlim(*_background_xlim(CURRENT_FOX_MASKED_ALL, CURRENT_FOXF1_Q995))
axes[0].legend(frameon=False, fontsize=8)

axes[1].hist(CURRENT_BMP_MASKED_ALL, bins=180, color="#a5d6a7", edgecolor="none", alpha=0.85, density=True, label="pooled within-cyst pixels")
axes[1].axvline(CURRENT_BMP4_Q995, color="#111111", linewidth=1.2, linestyle="--", label="global q99.5 cutoff")
axes[1].set_title("BMP4 pooled within-cyst corrected pixels")
axes[1].set_xlabel("BMP4 corrected intensity")
axes[1].set_ylabel("density")
axes[1].set_xlim(*_background_xlim(CURRENT_BMP_MASKED_ALL, CURRENT_BMP4_Q995))
axes[1].legend(frameon=False, fontsize=8)

fig.savefig(CURRENT_Q995_MASKED_HIST_FIG, dpi=220, bbox_inches="tight")
print("Saved:", CURRENT_Q995_MASKED_HIST_FIG)
plt.show()

            ## Review-Plane Overlay QC

            These are the current reporter-positive calls on the selected review plane for each stack.
            The raw panels use a slightly tighter contrast stretch than the generic notebook default so dim reporter structure is easier to read.
            The thresholded panels show only the positive pixels on black.
            

In [ ]:
def _robust_rescale(image: np.ndarray, q_low: float = 0.01, q_high: float = 0.995) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros_like(arr, dtype=np.float32)
    lo = float(np.quantile(finite, q_low))
    hi = float(np.quantile(finite, q_high))
    if not np.isfinite(lo):
        lo = 0.0
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0)


def _raw_preview_rescale(image: np.ndarray) -> np.ndarray:
    return _robust_rescale(image, q_low=0.02, q_high=0.99)


def _positive_pixels_only_view(raw_image: np.ndarray, positive_mask: np.ndarray) -> np.ndarray:
    view = np.zeros_like(np.asarray(raw_image, dtype=np.float32), dtype=np.float32)
    pos = np.asarray(positive_mask, dtype=bool)
    if np.any(pos):
        view[pos] = np.asarray(raw_image, dtype=np.float32)[pos]
    return _robust_rescale(view)


def plot_current_q995_overlay_grid() -> Path:
    fig, axes = plt.subplots(len(manifest_df), 5, figsize=(20, 3.6 * len(manifest_df)), constrained_layout=True)
    if len(manifest_df) == 1:
        axes = np.asarray([axes])

    for ax_row, row in zip(axes, manifest_df.itertuples(index=False)):
        stack = dqh.load_czi_stack(ROOT / row.file_path)
        z_index = int(row.selected_review_z_0based) if pd.notna(row.selected_review_z_0based) else int(stack.data_czyx.shape[1] // 2)
        metric = plane_metrics_df[
            (plane_metrics_df["file_id"] == int(row.file_id))
            & (plane_metrics_df["z_index"] == z_index)
        ].iloc[0]
        mask = tifffile.imread(ROOT / metric["mask_path"]).astype(bool)
        dapi = np.asarray(stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "dapi"), z_index], dtype=np.float32)
        fox_raw = np.asarray(stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "foxf1"), z_index], dtype=np.float32)
        bmp_raw = np.asarray(stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "bmp4"), z_index], dtype=np.float32)

        fox_corr = fox_raw - np.float32(CURRENT_BG_MEDIANS[int(row.file_id)]["foxf1"])
        bmp_corr = bmp_raw - np.float32(CURRENT_BG_MEDIANS[int(row.file_id)]["bmp4"])
        fox_pos = mask & np.isfinite(fox_corr) & (fox_corr > np.float32(CURRENT_FOXF1_Q995))
        bmp_pos = mask & np.isfinite(bmp_corr) & (bmp_corr > np.float32(CURRENT_BMP4_Q995))

        panels = [
            ("DAPI + mask", _robust_rescale(dapi), "gray"),
            ("FOXF1 raw", _raw_preview_rescale(fox_raw), "gray"),
            ("FOXF1 q99.5+ pixels only", _positive_pixels_only_view(fox_raw, fox_pos), "gray"),
            ("BMP4 raw", _raw_preview_rescale(bmp_raw), "gray"),
            ("BMP4 q99.5+ pixels only", _positive_pixels_only_view(bmp_raw, bmp_pos), "gray"),
        ]
        for ax, (title, img, cmap) in zip(ax_row, panels):
            ax.imshow(img, cmap=cmap)
            if title == "DAPI + mask":
                ax.contour(mask.astype(float), levels=[0.5], colors="#00e5ff", linewidths=0.8)
            ax.set_title(f"{row.position_label} | z{z_index + 1} | {title}")
            ax.axis("off")

        ax_row[2].set_xlabel(
            f"thr={CURRENT_FOXF1_Q995:.1f} | med={CURRENT_BG_MEDIANS[int(row.file_id)]['foxf1']:.0f} | "
            f"FOXF1+={100.0 * float(np.mean(fox_pos[mask])):.1f}%"
        )
        ax_row[4].set_xlabel(
            f"thr={CURRENT_BMP4_Q995:.1f} | med={CURRENT_BG_MEDIANS[int(row.file_id)]['bmp4']:.0f} | "
            f"BMP4+={100.0 * float(np.mean(bmp_pos[mask])):.1f}%"
        )

    fig.suptitle("Current state | per-image median subtraction + pooled off-cyst q99.5 cutoffs")
    fig.savefig(CURRENT_Q995_OVERLAY_FIG, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved:", CURRENT_Q995_OVERLAY_FIG)
    return CURRENT_Q995_OVERLAY_FIG


plot_current_q995_overlay_grid()